# Giải bài tập MCQ

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load dữ liệu

In [2]:
data_dir = '../Data/'
orders = pd.read_csv(data_dir + 'orders.csv', parse_dates=['order_date'], low_memory=False)
products = pd.read_csv(data_dir + 'products.csv', low_memory=False)
returns = pd.read_csv(data_dir + 'returns.csv', low_memory=False)
web_traffic = pd.read_csv(data_dir + 'web_traffic.csv', low_memory=False)
order_items = pd.read_csv(data_dir + 'order_items.csv', low_memory=False)
customers = pd.read_csv(data_dir + 'customers.csv', low_memory=False)
payments = pd.read_csv(data_dir + 'payments.csv', low_memory=False)
geography = pd.read_csv(data_dir + 'geography.csv', low_memory=False)
print('Đã tải xong toàn bộ dữ liệu.')

Đã tải xong toàn bộ dữ liệu.


### Q1: Trung vị số ngày giữa hai lần mua liên tiếp

Yêu cầu: Tính trung vị khoảng cách số ngày giữa hai lần mua hàng liên tiếp của các khách hàng có nhiều hơn 1 đơn.

In [3]:
# Sắp xếp theo khách hàng và ngày
orders_sorted = orders.sort_values(by=['customer_id', 'order_date'])

# Lọc khách hàng có nhiều hơn 1 đơn (Dùng duplicated để tối ưu hơn isin)
df_multi = orders_sorted[orders_sorted.duplicated(subset='customer_id', keep=False)].copy()

# Tính chênh lệch ngày giữa các đơn liên tiếp
df_multi['diff_days'] = df_multi.groupby('customer_id')['order_date'].diff().dt.days

# Tính trung vị
median_gap = df_multi['diff_days'].median()
options_q1 = {30: 'A', 90: 'B', 180: 'C', 365: 'D'}
ans_val = min(options_q1.keys(), key=lambda k: abs(k - median_gap))
print(f"Q1 Answer: {median_gap} ngày => Đáp án {options_q1[ans_val]}")

Giá trị trung vị: 144.0 ngày
==> Đáp án Q1: C


### Q2: Tỷ suất lợi nhuận gộp trung bình theo segment

Yêu cầu: Tìm phân khúc (segment) sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất.

In [4]:
# Tính lợi nhuận gộp: (Price - COGS) / Price
products['margin'] = (products['price'] - products['cogs']) / products['price']

# Tính trung bình theo segment
margin_by_segment = products.groupby('segment')['margin'].mean()
options_q2 = {'Premium': 'A', 'Performance': 'B', 'Activewear': 'C', 'Standard': 'D'}

top_segment = margin_by_segment.idxmax()
print(f"Đáp án Q2: {top_segment} => Đáp án {options_q2.get(top_segment, 'Không rõ')}")

Tỷ suất lợi nhuận gộp trung bình của các phân khúc trong đáp án:
 - Premium: 0.2854
 - Performance: 0.2636
 - Activewear: 0.2656
 - Standard: 0.3134

Phân khúc có tỷ suất lợi nhuận cao nhất toàn bộ dữ liệu: Standard (0.3134)
==> Đáp án Q2: D


### Q3: Lý do trả hàng nhiều nhất cho danh mục Streetwear

Yêu cầu: Xác định nguyên nhân trả hàng phổ biến nhất cho danh mục Streetwear.

In [5]:
# Merge returns với products để lấy category
ret_prod = returns.merge(products, on='product_id')

# Lọc danh mục Streetwear
streetwear_returns = ret_prod[ret_prod['category'] == 'Streetwear']

# Đếm tần suất các lý do trả hàng
top_reason = streetwear_returns['return_reason'].value_counts()
options_q3 = {'defective': 'A', 'wrong_size': 'B', 'changed_mind': 'C', 'not_as_described': 'D'}
top_reason_val = top_reason.index[0]
print(f"Đáp án Q3: {top_reason_val} => Đáp án {options_q3.get(top_reason_val, 'Không rõ')}")

Lý do trả hàng nhiều nhất: wrong_size
==> Đáp án Q3: B


### Q4: Nguồn traffic có tỷ lệ thoát trung bình thấp nhất

Yêu cầu: Nguồn lưu lượng (traffic_source) nào mang lại lượng khách có tỷ lệ thoát (bounce_rate) trung bình thấp nhất?

In [6]:
# Nhóm theo nguồn traffic và tính trung bình bounce_rate
bounce_rate = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
options_q4 = {'organic_search': 'A', 'paid_search': 'B', 'email_campaign': 'C', 'social_media': 'D'}
top_source = bounce_rate.index[0]
print(f"Đáp án Q4: {top_source} => Đáp án {options_q4.get(top_source, 'Không rõ')}")

Nguồn có tỷ lệ thoát thấp nhất: email_campaign
==> Đáp án Q4: C


### Q5: Tỷ lệ order_items có khuyến mãi

Yêu cầu: Tính tỷ lệ phần trăm số dòng đơn hàng (order_items) có áp dụng mã khuyến mãi.

In [7]:
# Đếm số lượng items có áp dụng promo_id hoặc promo_id_2
has_promo = (order_items['promo_id'].notnull() | order_items['promo_id_2'].notnull()).sum()
total_items = len(order_items)

# Tính tỷ lệ %
promo_ratio = has_promo / total_items * 100
options_q5 = {12: 'A', 25: 'B', 39: 'C', 54: 'D'}
ans_val = min(options_q5.keys(), key=lambda k: abs(k - promo_ratio))
print(f"Đáp án Q5: {promo_ratio:.2f}% => Đáp án {options_q5[ans_val]}")

Tỷ lệ có khuyến mãi: 38.66%
==> Đáp án Q5: C


### Q6: Nhóm tuổi có số đơn hàng trung bình cao nhất

Yêu cầu: Nhóm tuổi (age_group) nào có số đơn hàng trung bình mỗi khách hàng cao nhất?

In [8]:
# Đếm số đơn hàng của mỗi khách hàng
orders_per_cust = orders['customer_id'].value_counts().reset_index()
orders_per_cust.columns = ['customer_id', 'order_count']

# Merge với bảng khách hàng để lấy age_group
cust_orders = customers.merge(orders_per_cust, on='customer_id', how='left')
cust_orders['order_count'] = cust_orders['order_count'].fillna(0)

# Tính trung bình số đơn theo age_group
age_group_orders = cust_orders[cust_orders['age_group'].notnull()].groupby('age_group')['order_count'].mean().sort_values(ascending=False)
options_q6 = {'55+': 'A', '25-34': 'B', '35-44': 'C', '45-54': 'D'}
top_age_group = age_group_orders.index[0]
normalized_age_group = top_age_group.replace('–', '-')
print(f"Đáp án Q6: {top_age_group} => Đáp án {options_q6.get(normalized_age_group, 'Không rõ')}")

Nhóm tuổi có số đơn hàng TB cao nhất: 55+
==> Đáp án Q6: A


### Q7: Vùng có doanh thu cao nhất

Yêu cầu: Khu vực địa lý (region) nào tạo ra nhiều doanh thu nhất?

In [9]:
# Merge orders với customers (bỏ cột zip của orders để tránh trùng lặp khi merge)
orders_cust = orders.drop(columns=['zip']).merge(customers, on='customer_id', how='inner')
# Merge với geography để lấy region
orders_geo = orders_cust.merge(geography, on=['zip', 'city'], how='inner')
# Merge với payments để lấy doanh thu (payment_value)
orders_full = orders_geo.merge(payments, on='order_id', how='inner')

# Tính tổng doanh thu theo region
region_revenue = orders_full.groupby('region')['payment_value'].sum().sort_values(ascending=False)
options_q7 = {'West': 'A', 'Central': 'B', 'East': 'C'}
top_region = region_revenue.index[0]
print(f"Đáp án Q7: {top_region} => Đáp án {options_q7.get(top_region, 'D')}")

Vùng có doanh thu cao nhất: East
==> Đáp án Q7: C


### Q8: Phương thức thanh toán nhiều nhất trong đơn huỷ

Yêu cầu: Phương thức thanh toán nào phổ biến nhất trong các đơn hàng bị huỷ (cancelled)?

In [10]:
# Lọc các đơn bị hủy
cancelled_orders = orders[orders['order_status'] == 'cancelled']

# Đếm tần suất phương thức thanh toán
top_payment = cancelled_orders['payment_method'].value_counts()
options_q8 = {'credit_card': 'A', 'cod': 'B', 'paypal': 'C', 'bank_transfer': 'D'}
top_payment_method = top_payment.index[0]
print(f"Đáp án Q8: {top_payment_method} => Đáp án {options_q8.get(top_payment_method, 'Không rõ')}")

Phương thức thanh toán nhiều nhất khi huỷ đơn: credit_card


==> Đáp án Q8: A


### Q9: Kích thước sản phẩm có tỷ lệ trả hàng cao nhất

Yêu cầu: Kích thước (size) nào có tỷ lệ trả hàng (số dòng bị trả / số dòng đã bán) cao nhất?

In [11]:
# Đếm số lượng đã bán theo từng size
items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id')
size_counts = items_with_size['size'].value_counts()

# Đếm số lượng bị trả lại theo từng size
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')
return_counts = returns_with_size['size'].value_counts()

# Tính tỷ lệ trả hàng
return_ratio = (return_counts / size_counts).dropna().sort_values(ascending=False)
options_q9 = {'S': 'A', 'M': 'B', 'L': 'C', 'XL': 'D'}
top_size = return_ratio.index[0]
print(f"Đáp án Q9: {top_size} => Đáp án {options_q9.get(top_size, 'Không rõ')}")

Kích thước có tỷ lệ trả hàng cao nhất: S
==> Đáp án Q9: A


### Q10: Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất

Yêu cầu: Kế hoạch trả góp (installments) nào có giá trị thanh toán trung bình (payment_value) cao nhất?

In [12]:
# Tính trung bình giá trị thanh toán theo số tháng trả góp
avg_payment_installments = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
options_q10 = {1: 'A', 3: 'B', 6: 'C', 12: 'D'}
top_installments = avg_payment_installments.index[0]
print(f"Đáp án Q10: {top_installments} kỳ => Đáp án {options_q10.get(top_installments, 'Không rõ')}")

Kế hoạch trả góp có GT thanh toán TB cao nhất: 6 kỳ
==> Đáp án Q10: C
